# Question 14 - 134. Gas Station

There are `n` gas stations along a **circular** route, where the amount of gas at the `i-th` station is `gas[i]`.

You have a car with an unlimited gas tank and it costs `cost[i]` of gas to travel from the `i-th` station to its next `(i + 1)-th` station. You begin the journey with an empty tank at one of the gas stations.

Given two integer arrays `gas` and `cost`, return *the starting gas station's index if you can travel around the circuit once in the clockwise direction, otherwise return* `-1`. If there exists a solution, it is **guaranteed** to be **unique**.

**Example 1:**

    Input: gas = [1,2,3,4,5], cost = [3,4,5,1,2]
    Output: 3
    Explanation:
    Start at station 3 (index 3) and fill up with 4 unit of gas. Your tank = 0 + 4 = 4
    Travel to station 4. Your tank = 4 - 1 + 5 = 8
    Travel to station 0. Your tank = 8 - 2 + 1 = 7
    Travel to station 1. Your tank = 7 - 3 + 2 = 6
    Travel to station 2. Your tank = 6 - 4 + 3 = 5
    Travel to station 3. The cost is 5. Your gas is just enough to travel back to station 3.
    Therefore, return 3 as the starting index.

**Example 2:**

    Input: gas = [2,3,4], cost = [3,4,3]
    Output: -1
    Explanation:
    You can't start at station 0 or 1, as there is not enough gas to travel to the next station.
    Let's start at station 2 and fill up with 4 unit of gas. Your tank = 0 + 4 = 4
    Travel to station 0. Your tank = 4 - 3 + 2 = 3
    Travel to station 1. Your tank = 3 - 3 + 3 = 3
    You cannot travel back to station 2, as it requires 4 unit of gas but you only have 3.
    Therefore, you can't travel around the circuit once no matter where you start.

**Constraints:**

- `n == gas.length == cost.length`
- `1 <= n <= 10^5`
- `0 <= gas[i], cost[i] <= 10^4`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy (reset start when the running tank goes negative) + total-balance check

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(n²) | O(1) |
| 2️⃣ Optimized | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"At station `i` I gain `gas[i]` and then spend `cost[i]` to drive to `i+1`. So each station has a **net** value `diff[i] = gas[i] − cost[i]`. I need a starting point where the running sum of `diff` around the circle never goes negative."*

- Circular → after station `n−1` I go back to `0`. **Yes.**
- Answer unique if it exists → **Yes.** So I just need to *find* one valid start, without comparing candidates.
- Tank empty at the start; is reaching a station with exactly 0 OK? → **Yes.** Only a *negative* tank fails.
- Return `-1` if impossible.

### Step 1 · Brute force

**Idea:** Try each station as the start and simulate the full loop. Abandon that start as soon as the tank drops below zero.

🗣️ *"The direct approach simulates the trip from every starting station, which is O(n) starts × O(n) steps = O(n²). For n = 10⁵ that's 10¹⁰, too slow. But simulating will show me what information a failed attempt gives me."*

In [1]:
class SolutionBrute:
    def canCompleteCircuit(self, gas: list[int], cost: list[int]) -> int:
        n = len(gas)
        for start in range(n):
            tank = 0
            for k in range(n):
                i = (start + k) % n
                tank += gas[i] - cost[i]
                if tank < 0:
                    break
            else:                    # loop finished without break
                return start
        return -1

### Step 2 · Optimize

**Bottleneck:** after a failed start, brute force retries from `start + 1` and re-simulates stations it already knows are bad.

**Two key facts:**

1. **Feasibility = total balance.** If `sum(gas) < sum(cost)`, no start can work, because you can't produce gas from nowhere. If `sum(gas) ≥ sum(cost)`, a valid start **is guaranteed to exist**.

2. **Skip a whole failed stretch.** Suppose I start at `s` and first run dry when driving out of station `j`. Then **no station between `s` and `j` can be a valid start either.**
   *Why:* arriving at any middle station `k` from `s`, my tank was **≥ 0** (I hadn't failed yet). Starting fresh at `k` means arriving with tank **= 0**, which is no better. If I failed at `j` with a head start, I'll fail at `j` without it too.
   So the next candidate is `j + 1`.

So one left-to-right pass is enough: keep a running `tank`, and whenever it goes negative, set `start = i + 1` and reset `tank = 0`. If the total balance is non-negative, the last `start` is the answer.

🗣️ *"First, if total gas is less than total cost, it's impossible. Otherwise a start exists. Then I scan once with a running tank. If the tank goes negative at station i, then none of the stations from my current start up to i can work, because they'd arrive at i with even less gas. So I jump the start to i+1 and reset. The final start is the answer. One pass, O(1) space."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Simulate from every start | ❌ | O(n²). It re-simulates stretches already proven bad. |
| Double the array (`diff + diff`) and use a sliding window | ⚠️ | Works, but uses O(n) extra space and more index bookkeeping for no gain. |
| Prefix sums: start right after the **minimum** prefix sum | ✅ | Equivalent and elegant: the lowest point of the cumulative balance is where you should start. O(n), O(1). Good to mention. |
| **Greedy reset + total check** | ✅ | One pass, two integers. The skip lemma justifies jumping over whole failed segments. |

**Why don't we need to wrap around and check the stations before `start`?** Because the total balance is ≥ 0. Whatever deficit existed in `[0, start)` is covered by the surplus collected from `start` to the end. Since `total = (deficit part) + (surplus part) ≥ 0`, the surplus is at least as large as the deficit.

In [2]:
class Solution:
    def canCompleteCircuit(self, gas: list[int], cost: list[int]) -> int:
        total = 0      # overall balance: decides if any answer exists
        tank = 0       # balance since the current candidate start
        start = 0
        for i in range(len(gas)):
            diff = gas[i] - cost[i]
            total += diff
            tank += diff
            if tank < 0:           # can't leave station i from `start`
                start = i + 1      # skip every station in [start, i]
                tank = 0
        return start if total >= 0 else -1

### Step 3 · Toy example walkthrough

`gas = [1, 2, 3, 4, 5]`, `cost = [3, 4, 5, 1, 2]` → `diff = [−2, −2, −2, +3, +3]`

| i | diff | tank (after) | tank < 0? | start | total |
|---|---|---|---|---|---|
| 0 | −2 | −2 | ✅ reset | 1 | −2 |
| 1 | −2 | −2 | ✅ reset | 2 | −4 |
| 2 | −2 | −2 | ✅ reset | 3 | −6 |
| 3 | +3 | 3 | ❌ | 3 | −3 |
| 4 | +3 | 6 | ❌ | 3 | **0** |

`total = 0 ≥ 0` → answer **3** ✅. The +6 surplus from stations 3–4 exactly covers the −6 deficit from stations 0–2.

**Example 2:** `gas = [2,3,4]`, `cost = [3,4,3]` → `diff = [−1, −1, +1]`, total = −1 < 0 → **−1** ✅

In [3]:
cases = [
    ([1, 2, 3, 4, 5], [3, 4, 5, 1, 2], 3),
    ([2, 3, 4], [3, 4, 3], -1),
    ([5], [4], 0),
    ([4], [5], -1),
    ([3, 1, 1], [1, 2, 2], 0),
    ([1, 2], [2, 1], 1),
    ([0, 0, 0], [0, 0, 0], 0),
]
for gas, cost, expected in cases:
    assert SolutionBrute().canCompleteCircuit(gas, cost) == expected
    assert Solution().canCompleteCircuit(gas, cost) == expected
    print(gas, cost, "->", expected)

import random
for _ in range(1000):
    n = random.randint(1, 8)
    gas = [random.randint(0, 5) for _ in range(n)]
    cost = [random.randint(0, 5) for _ in range(n)]
    b = SolutionBrute().canCompleteCircuit(gas, cost)
    o = Solution().canCompleteCircuit(gas, cost)
    # with ties (non-unique answers) both must at least agree on feasibility
    assert (b == -1) == (o == -1), (gas, cost)
    if o != -1:
        assert SolutionBrute().canCompleteCircuit(gas[o:] + gas[:o], cost[o:] + cost[:o]) == 0
print("All tests passed ✅")

[1, 2, 3, 4, 5] [3, 4, 5, 1, 2] -> 3
[2, 3, 4] [3, 4, 3] -> -1
[5] [4] -> 0
[4] [5] -> -1
[3, 1, 1] [1, 2, 2] -> 0
[1, 2] [2, 1] -> 1
[0, 0, 0] [0, 0, 0] -> 0
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Brute force | O(n²) | O(1) | n starts × up to n steps |
| **Greedy reset** | **O(n)** | **O(1)** | Each failed segment is skipped entirely, so each station is visited once |

**Complexity, explained:** a single `for` loop over n stations, with O(1) arithmetic each. We store `total`, `tank`, and `start`, so space is O(1).

**The two proofs to have ready:**
1. *total ≥ 0 ⇒ a solution exists* (and total < 0 ⇒ none).
2. *Failing at `j` from `s` ⇒ every start in `(s, j]` also fails*, because each of them would arrive with a tank ≤ the one we had.

**Edge cases:** single station, all zeros (start 0), a station with `gas == cost`, a deficit only at the very end.

**Follow-up:** *What if solutions aren't unique and you need all of them?* → Use the prefix-sum view. Every index right after a **global minimum** of the cumulative balance is a valid start.

🗣️ **30-second recap:** *"Convert each station to net gain gas minus cost. If the total is negative, return −1. Otherwise scan once with a running tank. Whenever it dips below zero at i, no start between my candidate and i can work, so I move the candidate to i+1 and reset. The last candidate is the answer. O(n) time, O(1) space."*